In [ ]:
!pip install faiss-cpu sentence-transformers
!pip install underthesea
!pip install langchain
!pip install flagembedding
!pip install langchain_groq
!pip install -U langchain-community

In [2]:
import json

# Đọc file JSON
with open("D:\Legal_Document_Retrieval\corpus.json", 'r') as f:
    data = json.load(f)

# Tạo dictionary với phần tử thứ 2 làm key và phần tử thứ 3 làm value
# documents = {sublist[1]: (sublist[2], sublist[0]) for sublist in data}
documents = [sublist[0] for sublist in data]

KeyError: 0

In [ ]:
import os
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

def load_pkl_files(folder_path, start_idx=0, end_idx=1):
    embeddings = []
    for i in range(start_idx, end_idx + 1):
        filename = f"embeddings_{i}.pkl"
        file_path = os.path.join(folder_path, filename)
        
        if os.path.exists(file_path):
            with open(file_path, 'rb') as f:
                data = pickle.load(f)
                embeddings.append(data)  # Giả sử data chứa embedding numpy array
        else:
            print(f"File {filename} không tồn tại!")
    
    # Ghép tất cả các embedding lại thành một ma trận
    if embeddings:
        return np.vstack(embeddings)
    else:
        return None

    
# Tạo Faiss index với GPU
def create_faiss_index(embeddings, use_gpu=True):
    dim = embeddings.shape[1]  # Kích thước của embedding
    index = faiss.IndexFlatL2(dim)  # Sử dụng Faiss IndexFlatL2 cho tính toán cosine similarity
    
    if use_gpu:
        # Di chuyển index lên GPU
        res = faiss.StandardGpuResources()  # Sử dụng GPU resources
        index = faiss.index_cpu_to_gpu(res, 0, index)  # Chuyển index lên GPU (GPU số 0)
    
    # Thêm embedding vào index
    index.add(embeddings)
    return index

# Tìm kiếm với Faiss
def search_faiss_index(index, query_embedding, k=5):
    # Tính toán cosine similarity giữa query_embedding và các embedding trong index
    distances, indices = index.search(query_embedding, k)
    return distances, indices

In [ ]:
# Đường dẫn đến thư mục chứa các file .pkl
folder_path = '/kaggle/input/dvt-embedding/EMBEDDING'

# 1. Load toàn bộ embedding từ các file .pkl
embeddings = load_pkl_files(folder_path)

# 2. Tạo Faiss index với GPU
index = create_faiss_index(embeddings, use_gpu=True)

In [ ]:
from langchain.vectorstores import FAISS
from langchain.schema import Document
from pydantic import Field
from typing import List
from langchain.schema import BaseRetriever

class KeywordRetriever(BaseRetriever):
    documents: List[Document] = Field(default_factory=list)  # Khai báo `documents` trong Pydantic
    index: FAISS  # FAISS index phải được truyền vào

    def __init__(self, documents: List[Document], index: FAISS):
        """
        Custom Retriever sử dụng tìm kiếm keyword với FAISS index.
        Args:
            documents (List[Document]): Danh sách các tài liệu để tìm kiếm.
            index (FAISS): FAISS index chứa các embeddings của tài liệu.
        """
        super().__init__()
        self.documents = documents
        self.index = index

    def _get_relevant_documents(self, query: str) -> List[Document]:
        """
        Trả về danh sách các tài liệu phù hợp với query sử dụng FAISS.
        """
        # Tạo embedding cho câu truy vấn
        query_embedding = embeddings.embed_query(query)

        # Truy vấn FAISS để tìm các tài liệu tương tự nhất
        _, indices = self.index.search(query_embedding, k=5)  # k là số lượng tài liệu cần lấy

        # Trả về các tài liệu tương ứng với các chỉ mục
        relevant_docs = [self.documents[i] for i in indices[0]]
        return relevant_docs

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        """
        Phiên bản bất đồng bộ của _get_relevant_documents.
        """
        return self._get_relevant_documents(query)

